In [1]:
import pandas as pd
import os
import sys
dir1 = os.path.abspath(os.path.join(os.getcwd(), '../../analysisFunctions'))
sys.path.insert(0, dir1)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import re
from machine_learning import *
from hyperion_utils import *

In [2]:
myDfs = readDfs()

In [3]:
print(myDfs['Description_tables_variables'][['Tables', 'Variable', 'LabelTranslated', 'Label']].to_string())

         Tables                          Variable                                                                                       LabelTranslated                                                                                                 Label
0           ADL                            SUBJID                                                                      Subject Identifier for the Study                                                                      Subject Identifier for the Study
1           ADL                             ADL_0                                                                      Le score ADL a t-il été complété                                                                      Le score ADL a t-il été complété
2           ADL                        ADL_1_TEMP                                                  Hygiène corporelle (champ intermediaire pour saisie)                                                  Hygiène corporelle (champ intermediai

In [4]:
bio_data_columns = list(myDfs['Description_tables_variables']['Variable'].iloc[52:76])
bio_data_columns.append('SUBJID')
bio_data_columns_descr = list(myDfs['Description_tables_variables']['Label'].iloc[52:76])
bio_visit_column = 'VISIT'
cpc_column = 'CPC_SC3'
group_colums = ['V0_BRAS2', 'groupe']
ds_columns = 'DS_DATA_REFUS'
ecg_columns = list(myDfs['Description_tables_variables']['Variable'].iloc[137:154])
ecg_columns_descr = list(myDfs['Description_tables_variables']['LabelTranslated'].iloc[137:154])
j0_drop_columns = list(myDfs['Description_tables_variables']['Variable'].iloc[362:384])
j0_reflex_columns = list(myDfs['Description_tables_variables']['Variable'].iloc[273:285])
j0_reflex_columns_descr = list(myDfs['Description_tables_variables']['LabelTranslated'].iloc[273:285])
sofa_columns = list(myDfs['Description_tables_variables']['Variable'].iloc[586:593])
sofa_columns.append('SUBJID')
ei_columns = list(myDfs['Description_tables_variables']['Variable'].iloc[158:168])
ei_columns = ei_columns + (['SUBJID', 'EI_ARYTHMI', 'EI_ANTIEPILEPTIQ'])

In [5]:
myPredictorsDf = myDfs['ds'][myDfs['ds']['DS_DATA_REFUS'] != 1][['SUBJID', 'DS_DATA_REFUS']]
myPredictorsDf = myPredictorsDf.merge(myDfs['bras_rando'], on='SUBJID')
myPredictorsDf = myPredictorsDf[~myPredictorsDf['groupe'].isna()]
myPredictorsDf = myPredictorsDf.merge(myDfs['cpc'][['SUBJID', cpc_column]], on='SUBJID')
myPredictorsDf = myPredictorsDf.merge(myDfs['j0'], on='SUBJID')
myPredictorsDf = myPredictorsDf.merge(myDfs['bio'][myDfs['bio']['VISIT'] == 'J0'][bio_data_columns], on='SUBJID')
myPredictorsDf = myPredictorsDf.merge(myDfs['ecg'][myDfs['ecg']['VISIT'] == 'J0'][ecg_columns], on='SUBJID')
myPredictorsDf = myPredictorsDf.merge(myDfs['sofa'][myDfs['sofa']['VISIT'] == 'J0'][sofa_columns], on='SUBJID')
myPredictorsDf = myPredictorsDf.merge(myDfs['ei'][myDfs['ei']['CRFSTATUS'] == 'J0'][ei_columns], on='SUBJID')
# myPredictorsDf = myPredictorsDf.merge(myDfs['adl'][['SUBJID', 'ADL_SC']], on='SUBJID')
myPredictorsDf = myPredictorsDf.merge(myDfs['barthel'][['SUBJID', 'BARTHEL_SC']], on='SUBJID')
# myPredictorsDf = myPredictorsDf.merge(myDfs['mms'][['SUBJID', 'MMS_SC']], on='SUBJID')
myDay0SofaSc = myDfs['sofa'][(myDfs['sofa']['VISIT'] == 'J0')][['SUBJID', 'SOFA_SC']]
myDay7SofaSc = myDfs['sofa'][(myDfs['sofa']['VISIT'] == 'J7')][['SUBJID', 'SOFA_SC']]
myDay7SofaSc.rename(columns={'SOFA_SC': 'SOFA_SC7'}, inplace=True)
myDay0SofaSc.rename(columns={'SOFA_SC': 'SOFA_SC1'}, inplace=True)
myPredictorsDf = myPredictorsDf.merge(myDay7SofaSc, on='SUBJID')
myPredictorsDf = myPredictorsDf.merge(myDay0SofaSc, on='SUBJID')
myPredictorsDf = myPredictorsDf.merge(myDfs['ds'][['DS_DC', 'DS_DT_DC', 'SUBJID']], on='SUBJID')
myPredictorsDf = myPredictorsDf.merge(myDfs['date'][['DT_J0', 'SUBJID']], on='SUBJID')
myPredictorsDf['DAYS_ALIVE_30'] = (pd.to_datetime(myPredictorsDf['DS_DT_DC'], format='%d/%m/%Y') - pd.to_datetime(pd.to_datetime(myPredictorsDf['DT_J0']).dt.date)).dt.days
myPredictorsDf['DAYS_ALIVE_30'].fillna(30, inplace=True)
myPredictorsDf.loc[myPredictorsDf['DAYS_ALIVE_30'] > 30, 'DAYS_ALIVE_30'] = 30

/tmp/ipykernel_3218614/3756268822.py:21: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  myPredictorsDf['DAYS_ALIVE_30'] = (pd.to_datetime(myPredictorsDf['DS_DT_DC'], format='%d/%m/%Y') - pd.to_datetime(pd.to_datetime(myPredictorsDf['DT_J0']).dt.date)).dt.days
/tmp/ipykernel_3218614/3756268822.py:22: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  myPredictorsDf['DAYS_ALIVE_30'].fillna(30, inplace=True)


In [6]:
myPredictorsDf.drop(columns=['DS_DT_DC', 'DT_J0', 'DS_DATA_REFUS', 'Unnamed: 0_x', 'Unnamed: 0_y', 'V0_BRAS2', 'DATEDERANDOAPRENDREENCOMPTEPOURA', 'J0_PANCARTE', 'J0_PANCARTET1', 'J0_NAIS_M', 'J0_NAIS_A', 'J0_DNAIS',  'J0_CAUSE2_ACR_P', 'J0_H_37T1', 'VISIT'], inplace=True)
myPredictorsDf.drop(columns=j0_drop_columns, inplace=True)
myPredictorsDf['CPC_SC3'].fillna(5, inplace=True)

/tmp/ipykernel_3218614/199694927.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  myPredictorsDf['CPC_SC3'].fillna(5, inplace=True)


In [7]:
myPredictorsDf['CPC12'] = ((myPredictorsDf['CPC_SC3'] == 1) | (myPredictorsDf['CPC_SC3'] == 2)).astype(int)
myPredictorsDf['SEX'] = (myPredictorsDf['J0_SEX'] == 'Homme').astype(int)


In [8]:
myPredictorsDf.to_csv('predictorsDf.csv', index=False)

In [11]:
[x for x in myPredictorsDf.columns]

['SUBJID',
 'groupe',
 'CPC_SC3',
 'J0_SEX',
 'J0_TAILLE',
 'J0_POIDS',
 'J0_BMI',
 'J0_AGE',
 'J0_PAS',
 'J0_PAD',
 'J0_PAM',
 'J0_FC',
 'J0_SPO2',
 'J0_GLASGOW',
 'J0_GLASGOW_CONTROLE',
 'J0_OCULAIRE',
 'J0_VERBALE',
 'J0_MOTRICE',
 'J0_PUPILG',
 'J0_PUPILG_REA',
 'J0_PUPILD',
 'J0_PUPILD_REA',
 'J0_CILIAIRE',
 'J0_CORNEEN',
 'J0_REFLEXVEST',
 'J0_REFLEXCEPH',
 'J0_REFLEXCARD',
 'J0_TEMP',
 'J0_IGSII',
 'J0_MCCABE',
 'J0_KNAUS',
 'J0_CHARLSON1',
 'J0_CHARLSON2',
 'J0_CHARLSON3',
 'J0_CHARLSON4',
 'J0_CHARLSON5',
 'J0_CHARLSON6',
 'J0_CHARLSON7',
 'J0_CHARLSON8',
 'J0_CHARLSON9',
 'J0_CHARLSON10',
 'J0_CHARLSON11',
 'J0_CHARLSON12',
 'J0_CHARLSON13',
 'J0_CHARLSON14',
 'V0_CHARLSON15',
 'V0_CHARLSON16',
 'V0_CHARLSON17',
 'V0_CHARLSON18',
 'V0_CHARLSON18B',
 'V0_CHARLSON19',
 'J0_CHARLSON',
 'J0_ATCD',
 'J0_CARDIO',
 'J0_NYHA',
 'J0_MYOCARD',
 'J0_ARTERIO',
 'J0_HTA',
 'J0_POUMON',
 'J0_IRC',
 'J0_HYPERCAP',
 'J0_O2',
 'J0_TABAC',
 'J0_RYTHM',
 'J0_CAUSE2_ACR',
 'J0_DSA',
 'J0_DSA_P',